# ComicAnalizer - Pipeline Completo En Colab

Notebook oficial unico para procesar un dataset limpio de comics en Colab.

Flujo:

1. Clona ComicAnalizer.
2. Instala dependencias de Magi.
3. Sube un ZIP tipo `magi_clean_full.zip` con estructura `magi_clean_full/by_comic/<comic>/test_1_clean/*.jpg`.
4. Ejecuta Magi con `--task both`: detecciones + OCR propio de Magi.
5. Genera reporte de calidad Magi.
6. Instala PaddleOCR limpio despues de Magi.
7. Ejecuta PaddleOCR complementario sobre las mismas paginas.
8. Genera visualizaciones OCR simples y agrupadas para comparar Magi vs PaddleOCR.
9. Exporta evidencia OCR para calibracion/entrenamiento.
10. Genera page understanding: numeracion, tipo de pagina y reporte HTML.
11. Descarga un unico ZIP con todo el run.

Importante: usa GPU en Colab antes de empezar: `Runtime > Change runtime type > GPU`.

In [ ]:
# 1) Clonar repositorio limpio
!rm -rf /content/ComicAnalizer
!git clone https://github.com/nicolas4432/ComicAnalizer.git /content/ComicAnalizer
%cd /content/ComicAnalizer
!git log --oneline -5

In [ ]:
# 2) Verificar GPU sin importar torch en el kernel
import subprocess

print("GPU visible para Colab:")
nvidia = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
print(nvidia.stdout if nvidia.returncode == 0 else "nvidia-smi no disponible")
if nvidia.returncode != 0:
    raise RuntimeError("Activa GPU en Colab antes de continuar.")

In [ ]:
# 3) Instalar dependencias Magi
%cd /content/ComicAnalizer

import subprocess
import sys

commands = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.49.0",
        "huggingface_hub<1.0",
        "timm",
        "einops",
        "pytorch-metric-learning",
        "shapely",
        "Pillow",
        "opencv-python-headless",
    ],
]

for cmd in commands:
    print("Ejecutando:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("Verificando torch/Magi runtime en subprocess...")
subprocess.run([
    sys.executable, "-c",
    "import torch; print('torch=', torch.__version__); print('cuda=', torch.cuda.is_available()); print('gpu=', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"
], check=True)

## Subir Dataset

Sube **un ZIP** con estructura:

```text
magi_clean_full/
  by_comic/
    <comic_id>/
      test_1_clean/
        000.jpg
        001.jpg
```

Si Colab renombra el archivo como `(1)` o `(2)`, el notebook detecta la carpeta real autom?ticamente.

In [ ]:
# 4) Subir y descomprimir dataset limpio
from google.colab import files
from pathlib import Path

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]
if not zip_names:
    raise RuntimeError("Debes subir un ZIP de dataset limpio.")

DATASET_ZIP = zip_names[0]
print("DATASET_ZIP:", DATASET_ZIP)

!rm -rf /content/magi_sample
!mkdir -p /content/magi_sample
!unzip -q -o "$DATASET_ZIP" -d /content/magi_sample

by_comic_candidates = sorted(Path("/content/magi_sample").glob("**/by_comic"))
print("Carpetas by_comic encontradas:")
for candidate in by_comic_candidates:
    print("-", candidate)
if not by_comic_candidates:
    raise RuntimeError("No encontre by_comic dentro del ZIP.")
IMAGE_ROOT = str(by_comic_candidates[0])
print("IMAGE_ROOT:", IMAGE_ROOT)

In [ ]:
# 5) Configurar corrida completa
from pathlib import Path
import shlex
import subprocess

RUN_NAME = "colab_full_pipeline"
DATASET_NAME = "test_1_clean"
COMIC_ID = ""  # "" = todos; ejemplo: "nekkorarekko"
MAX_COMICS = 0  # 0 = todos
MAGI_TASK = "both"  # both = detecciones + OCR propio de Magi

RUN_ROOT = f"outputs/runs/{RUN_NAME}"
MAGI_OUTPUT = f"{RUN_ROOT}/magi"
MAGI_VISUALS = f"{RUN_ROOT}/visuals/magi_boxes"
ANALYSIS_OUTPUT = f"{RUN_ROOT}/analysis/magi_analysis_report.json"
PADDLE_OCR_OUTPUT = f"{RUN_ROOT}/analysis/paddle_magi_ocr_comparison.json"
PADDLE_OCR_VISUALS = f"{RUN_ROOT}/visuals/ocr_boxes"
PADDLE_OCR_GROUP_VISUALS = f"{RUN_ROOT}/visuals/ocr_groups"
OCR_EVIDENCE_OUTPUT = f"{RUN_ROOT}/analysis/ocr_evidence"
PAGE_UNDERSTANDING_OUTPUT = f"{RUN_ROOT}/analysis/page_understanding_report.json"
HTML_REPORT_OUTPUT = f"{RUN_ROOT}/report/index.html"
MAGI_CACHE = "outputs/cache/magi"

print("RUN_ROOT:", RUN_ROOT)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("MAGI_OUTPUT:", MAGI_OUTPUT)
print("PADDLE_OCR_VISUALS:", PADDLE_OCR_VISUALS)
print("PADDLE_OCR_GROUP_VISUALS:", PADDLE_OCR_GROUP_VISUALS)
print("PAGE_UNDERSTANDING_OUTPUT:", PAGE_UNDERSTANDING_OUTPUT)
print("HTML_REPORT_OUTPUT:", HTML_REPORT_OUTPUT)

In [ ]:
# 6) Ejecutar Magi: detecciones + OCR Magi
cmd = [
    "python", "-u", "-m", "tools.inspect_magi_dataset",
    "--input", IMAGE_ROOT,
    "--output-dir", MAGI_OUTPUT,
    "--visual-output-dir", MAGI_VISUALS,
    "--all-pages-per-comic",
    "--dataset-name", DATASET_NAME,
    "--task", MAGI_TASK,
    "--cache-dir", MAGI_CACHE,
    "--device", "cuda",
    "--dtype", "float16",
    "--max-comics", str(MAX_COMICS),
    "--no-panel-crops",
]
if COMIC_ID:
    cmd.extend(["--comic-id", COMIC_ID])

print("Ejecutando Magi:")
print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 7) Reporte de calidad Magi
!python -m tools.analyze_magi_results \
  --input "$MAGI_OUTPUT" \
  --output "$ANALYSIS_OUTPUT" \
  --top-n 20

import json
from pathlib import Path
magi_report = json.loads(Path(ANALYSIS_OUTPUT).read_text(encoding="utf-8"))
print(json.dumps(magi_report["summary"], indent=2, ensure_ascii=False))

In [ ]:
# 8) Instalar PaddleOCR limpio DESPUES de Magi
# Magi ya termin?. Ahora removemos torch para evitar el choque libtorch_cuda/NCCL visto en Colab.
%cd /content/ComicAnalizer

import os
import subprocess
import sys

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"
os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["PYTHONFAULTHANDLER"] = "1"

commands = [
    [sys.executable, "-m", "pip", "uninstall", "-y", "paddlepaddle", "paddlepaddle-gpu", "paddleocr", "paddlex", "torch", "torchvision", "torchaudio"],
    [
        sys.executable, "-m", "pip", "install", "-q",
        "shapely", "pyclipper", "opencv-python-headless", "Pillow", "numpy",
        "langchain", "langchain-core", "langchain-community", "langchain-text-splitters",
    ],
    [
        sys.executable, "-m", "pip", "install", "-q",
        "paddlepaddle-gpu==3.2.2",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
    ],
    [sys.executable, "-m", "pip", "install", "-q", "paddleocr==3.5.0"],
]

for cmd in commands:
    print("Ejecutando:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("Verificando PaddleOCR...")
import paddle
import paddleocr
print("paddle:", paddle.__version__)
print("paddleocr:", getattr(paddleocr, "__version__", "unknown"))
print("paddle cuda disponible:", paddle.is_compiled_with_cuda())
try:
    print("gpu count:", paddle.device.cuda.device_count())
except Exception as exc:
    print("gpu count no disponible:", exc)
if not paddle.is_compiled_with_cuda():
    raise RuntimeError("Paddle quedo sin CUDA. Reinicia runtime y revisa GPU.")
print("PaddleOCR importado correctamente")

In [ ]:
# 9) Ejecutar PaddleOCR complementario sobre todas las paginas Magi
import os
import shlex
import subprocess
import sys

PADDLE_OCR_LIMIT = 0  # 0 = todas
PADDLE_OCR_SELECTION = "first"

env = os.environ.copy()
env.update({
    "FLAGS_use_mkldnn": "0",
    "FLAGS_enable_pir_api": "0",
    "PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT": "0",
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
    "PYTHONFAULTHANDLER": "1",
})

cmd = [
    sys.executable, "-u", "-m", "tools.compare_magi_paddleocr",
    "--magi-input", MAGI_OUTPUT,
    "--image-root", IMAGE_ROOT,
    "--dataset-name", DATASET_NAME,
    "--selection", PADDLE_OCR_SELECTION,
    "--limit", str(PADDLE_OCR_LIMIT),
    "--seed", "42",
    "--lang", "en",
    "--visual-output-dir", PADDLE_OCR_VISUALS,
    "--grouped-visual-output-dir", PADDLE_OCR_GROUP_VISUALS,
    "--output", PADDLE_OCR_OUTPUT,
    "--checkpoint-every", "1",
]
if COMIC_ID:
    cmd.extend(["--comic-id", COMIC_ID])

print("Ejecutando PaddleOCR complementario:")
print(" ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, env=env)
if result.returncode != 0:
    raise RuntimeError(f"PaddleOCR fallo con codigo {result.returncode}")

In [ ]:
# 10) Resumen OCR, evidencia y reporte visual de page understanding
import json
from pathlib import Path
import subprocess
import shlex

ocr_report = json.loads(Path(PADDLE_OCR_OUTPUT).read_text(encoding="utf-8"))
print(json.dumps(ocr_report["summary"], indent=2, ensure_ascii=False))
for item in ocr_report["comparisons"][:10]:
    print(
        item["comic_id"], item["file_name"],
        "Magi=", item["magi_text_regions"],
        "Paddle=", item["paddle_text_blocks"],
        "match=", item["matched_regions"],
        "t=", round(item["paddle_elapsed_seconds"], 2),
        "err=", item["paddle_error"],
    )

cmd = [
    "python", "-m", "tools.export_ocr_evidence",
    "--ocr-report", PADDLE_OCR_OUTPUT,
    "--magi-input", MAGI_OUTPUT,
    "--image-root", IMAGE_ROOT,
    "--dataset-name", DATASET_NAME,
    "--output-dir", OCR_EVIDENCE_OUTPUT,
    "--asset-policy", "priority",
    "--max-asset-blocks", "500",
]
print("Exportando evidencia OCR:")
print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

cmd = [
    "python", "-m", "tools.generate_run_report",
    "--run-dir", RUN_ROOT,
    "--output-json", PAGE_UNDERSTANDING_OUTPUT,
    "--output-html", HTML_REPORT_OUTPUT,
    "--image-root", IMAGE_ROOT,
    "--dataset-name", DATASET_NAME,
]
print("Generando page understanding + HTML:")
print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 11) Descargar salida estandar completa, incluyendo report/index.html
from google.colab import files

zip_out = f"{RUN_NAME}_outputs.zip"
!zip -qr "$zip_out" "$RUN_ROOT" "$MAGI_CACHE"
print("Descargando:", zip_out)
files.download(zip_out)